# Supervised Learning Part 1 — Tutorial

**Primary outcome:** Train and compare Naive Bayes, K-Nearest Neighbors, Support Vector Machines, and Decision Trees on a real medical dataset, understanding the strengths and trade-offs of each.

**Time:** 75–90 minutes

---

## What we'll cover

| Section | Algorithm | Key concept |
|---------|-----------|-------------|
| 1 | Setup | Load real data, split, scale |
| 2 | Naive Bayes | Probabilistic classification |
| 3 | K-Nearest Neighbors | Similarity voting |
| 4 | Support Vector Machines | Maximum-margin hyperplane |
| 5 | Decision Trees | Recursive feature splits |
| 6 | Algorithm Comparison | Head-to-head benchmark |
| 7 | Hyperparameter Tuning Preview | GridSearchCV intro |
| 8 | Practice Exercises | Hands-on challenges |

We will use the **Wisconsin Breast Cancer** dataset throughout — a classic binary classification problem where we predict whether a tumour is malignant or benign from 30 numerical features derived from cell nucleus measurements. Using the same dataset for every algorithm lets us make fair, direct comparisons.

## Section 1 — Setup

We start by importing all libraries we need for the entire notebook, loading the dataset, and preparing a clean train/test split.

### Why scale features?
Some algorithms (KNN, SVM) are sensitive to the *magnitude* of features. Without scaling, a feature measured in thousands (e.g. area) will dominate over one measured in fractions (e.g. smoothness). We fit the scaler **only on the training set** to avoid data leakage — then apply the same transformation to the test set.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# ── Reproducibility & style ───────────────────────────────────────────────────
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print("All libraries imported successfully.")

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
data = load_breast_cancer()

X = data.data
y = data.target          # 0 = malignant, 1 = benign
feature_names = data.feature_names
class_names   = data.target_names   # ['malignant', 'benign']

print(f"Dataset: Wisconsin Breast Cancer")
print(f"  Samples  : {X.shape[0]}")
print(f"  Features : {X.shape[1]}")
print(f"  Classes  : {len(class_names)} — {list(class_names)}")
print()
print(f"Class distribution:")
for i, name in enumerate(class_names):
    count = (y == i).sum()
    print(f"  {name:12s}: {count} samples ({count/len(y)*100:.1f}%)")

In [ ]:
# ── Train / test split (80 / 20, stratified) ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Feature scaling — fit on train ONLY ──────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")
print()
print("Feature scaling applied (StandardScaler fitted on training data only).")

In [ ]:
# ── Helper: evaluate_model ────────────────────────────────────────────────────
# We call this function after fitting each algorithm. It prints a concise
# summary and returns a dict we collect for the final comparison table.

results_log = []   # accumulate results from every algorithm

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Fit, score, and log a classifier.
    
    Parameters
    ----------
    name  : human-readable algorithm name
    model : unfitted sklearn estimator
    X_tr, X_te, y_tr, y_te : train/test arrays
    
    Returns
    -------
    dict with name, train_acc, test_acc, fit_time, fitted model
    """
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    fit_time = time.perf_counter() - t0

    train_acc = accuracy_score(y_tr, model.predict(X_tr))
    test_acc  = accuracy_score(y_te, model.predict(X_te))

    print(f"{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Train accuracy : {train_acc:.4f}")
    print(f"  Test  accuracy : {test_acc:.4f}")
    print(f"  Fit time       : {fit_time*1000:.1f} ms")
    print()

    result = {
        'Algorithm' : name,
        'Train Acc' : round(train_acc, 4),
        'Test Acc'  : round(test_acc,  4),
        'Fit Time'  : round(fit_time * 1000, 2),
        'model'     : model
    }
    results_log.append(result)
    return result

print("evaluate_model() helper defined and ready.")

---
## Section 2 — Naive Bayes

### How it works

Naive Bayes applies **Bayes' theorem** with a strong ("naive") assumption: every feature is **conditionally independent** given the class label.

$$P(\text{class} \mid \text{features}) \;\propto\; P(\text{class}) \times \prod_{i} P(\text{feature}_i \mid \text{class})$$

Despite the unrealistic independence assumption, Naive Bayes often works surprisingly well in practice — especially when:
- Features are roughly independent (or correlations are weak)
- Training data is limited
- You need very fast training and prediction

**GaussianNB** assumes each feature follows a Gaussian (normal) distribution within each class. It estimates the mean and variance of each feature per class from the training data.

### Medical context
For cancer diagnosis, Naive Bayes is useful as a **fast baseline**. The model outputs calibrated probabilities, which clinicians can use to triage patients — e.g. "flag all predictions with malignant probability > 0.3 for review".

In [ ]:
# ── Fit Naive Bayes ───────────────────────────────────────────────────────────
# Note: GaussianNB does NOT require scaled features because it models
# each feature's distribution independently. We use the raw split here.

nb_model = GaussianNB()
nb_result = evaluate_model(
    'Naive Bayes', nb_model,
    X_train, X_test, y_train, y_test
)

# Detailed classification report
print(classification_report(y_test, nb_model.predict(X_test), target_names=class_names))

In [ ]:
# ── Visualise: predicted-probability histogram ────────────────────────────────
# predict_proba returns [P(malignant), P(benign)] for each sample.
# Plotting the histogram of P(benign) for each true class shows how
# well-separated the two distributions are.

probs = nb_model.predict_proba(X_test)[:, 1]   # P(benign)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: probability histogram by true class
for label, name, color in zip([0, 1], class_names, ['#e74c3c', '#2ecc71']):
    axes[0].hist(
        probs[y_test == label],
        bins=20, alpha=0.65, label=name, color=color, edgecolor='white'
    )
axes[0].axvline(0.5, ls='--', color='black', lw=1.5, label='decision boundary')
axes[0].set_xlabel('Predicted P(benign)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Naive Bayes — Predicted Probability Distribution', fontsize=13)
axes[0].legend()

# Right: confusion matrix
cm = confusion_matrix(y_test, nb_model.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Naive Bayes — Confusion Matrix', fontsize=13)

plt.tight_layout()
plt.show()

### Interpretation

The probability histogram shows that Naive Bayes is reasonably confident in most predictions — most test samples land near 0 (malignant) or 1 (benign) rather than near 0.5. The two distributions overlap only slightly, indicating good class separation.

### Naive Bayes — Pros & Cons

| Pros | Cons |
|------|------|
| Extremely fast to train and predict | Independence assumption is often violated |
| Works well with small training sets | Can underperform when features are correlated |
| Returns calibrated probabilities | Requires features to follow assumed distribution |
| Handles high-dimensional data naturally | Zero-frequency problem (smoothing needed for sparse data) |
| Interpretable: just means and variances per class | Less powerful than modern ensembles |

---
## Section 3 — K-Nearest Neighbors (KNN)

### How it works

KNN is a **lazy learner** — it memorises the training data and makes predictions by **similarity voting**:

1. Compute the distance from the new point to every training point (Euclidean by default)
2. Find the *k* nearest neighbours
3. Predict the majority class among those *k* neighbours

There is no explicit training phase. The entire workload shifts to prediction time.

### The k trade-off

- **k = 1** — the model memorises the training set perfectly → **overfitting**
- **k = N** — every point is predicted as the majority class → **underfitting**
- **Optimal k** sits somewhere in between and is found by measuring test accuracy across a range

### Why scaling matters for KNN
KNN uses raw distances. Without scaling, features with large magnitudes dominate the distance calculation. **Always scale before KNN.**

In [ ]:
# ── Fit KNN with k=5 ──────────────────────────────────────────────────────────
# We use scaled features for KNN.

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_result = evaluate_model(
    'KNN (k=5)', knn_model,
    X_train_scaled, X_test_scaled, y_train, y_test
)

print(classification_report(y_test, knn_model.predict(X_test_scaled), target_names=class_names))

In [ ]:
# ── Accuracy vs k — finding the U-shaped curve ────────────────────────────────
k_range      = range(1, 31)
train_accs   = []
test_accs    = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_accs.append(accuracy_score(y_train, knn.predict(X_train_scaled)))
    test_accs.append(accuracy_score(y_test,  knn.predict(X_test_scaled)))

optimal_k    = list(k_range)[np.argmax(test_accs)]
best_test_acc = max(test_accs)

print(f"Optimal k: {optimal_k}  (test accuracy = {best_test_acc:.4f})")

# ── Plot ──────────────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 5))
plt.plot(k_range, train_accs, 'o-', label='Train accuracy', color='#3498db', lw=2)
plt.plot(k_range, test_accs,  's-', label='Test accuracy',  color='#e67e22', lw=2)
plt.axvline(optimal_k, ls='--', color='#2ecc71', lw=1.8, label=f'Optimal k = {optimal_k}')
plt.xlabel('k (number of neighbours)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('KNN — Accuracy vs k\n(low k = overfit, high k = underfit)', fontsize=13)
plt.legend()
plt.ylim(0.88, 1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrix for optimal KNN ─────────────────────────────────────────
knn_best = KNeighborsClassifier(n_neighbors=optimal_k)
knn_best.fit(X_train_scaled, y_train)
y_pred_knn = knn_best.predict(X_test_scaled)

cm_knn = confusion_matrix(y_test, y_pred_knn)
disp   = ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=class_names)
disp.plot(colorbar=False, cmap='Oranges')
plt.title(f'KNN (k={optimal_k}) — Confusion Matrix', fontsize=13)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_knn, target_names=class_names))

### Interpretation

The accuracy-vs-k plot shows the classic bias-variance trade-off:
- At **k=1** the model perfectly memorises training data (100% train acc) but generalises poorly
- As k increases, train accuracy drops but test accuracy rises — the model becomes less sensitive to noise
- Beyond the optimal k, both train and test accuracy start to fall as the model becomes too smooth

### KNN — Pros & Cons

| Pros | Cons |
|------|------|
| Simple and intuitive | Slow at prediction time (scans all training points) |
| No training phase | High memory usage (stores entire training set) |
| Naturally handles multi-class | Sensitive to irrelevant features |
| Non-parametric — captures complex boundaries | Must scale features |
| Easy to explain to stakeholders | Curse of dimensionality with many features |

---
## Section 4 — Support Vector Machines (SVM)

### How it works

SVM finds the **maximum-margin hyperplane** — the decision boundary that maximises the gap between the two classes. The training points closest to the boundary are called **support vectors**; they alone define the boundary.

#### The kernel trick
Real data is rarely linearly separable. The **kernel trick** implicitly maps data to a higher-dimensional space where a linear boundary can separate the classes:

| Kernel | Best for | Boundary shape |
|--------|----------|----------------|
| `linear` | Linearly separable data, text classification | Straight hyperplane |
| `rbf` (Gaussian) | General-purpose, unknown structure | Radial / circular regions |
| `poly` | Polynomial relationships | Curved polynomial boundary |

#### Why SVM suits medical diagnosis
- Works well in **high-dimensional spaces** (30 features here)
- **Robust to outliers** — only support vectors matter
- Strong generalisation with the right kernel + regularisation

In [ ]:
# ── Fit SVM with RBF kernel ───────────────────────────────────────────────────
# probability=True enables predict_proba (needed for some metrics)

svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_result = evaluate_model(
    'SVM (RBF)', svm_model,
    X_train_scaled, X_test_scaled, y_train, y_test
)

print(classification_report(y_test, svm_model.predict(X_test_scaled), target_names=class_names))

In [ ]:
# ── Compare all three kernels ─────────────────────────────────────────────────
kernels = ['linear', 'rbf', 'poly']
kernel_results = {}

print("Kernel comparison on breast cancer dataset:")
print(f"{'Kernel':<10}  {'Train Acc':>10}  {'Test Acc':>10}  {'Fit Time (ms)':>14}")
print("-" * 50)

for k in kernels:
    t0  = time.perf_counter()
    clf = SVC(kernel=k, probability=True, random_state=42)
    clf.fit(X_train_scaled, y_train)
    ft  = (time.perf_counter() - t0) * 1000
    tr  = accuracy_score(y_train, clf.predict(X_train_scaled))
    te  = accuracy_score(y_test,  clf.predict(X_test_scaled))
    kernel_results[k] = {'model': clf, 'test_acc': te}
    print(f"{k:<10}  {tr:>10.4f}  {te:>10.4f}  {ft:>14.1f}")

In [ ]:
# ── Decision boundary with 2 features (mean radius vs mean texture) ───────────
# Using 2 features lets us draw the boundary on a 2-D plot.
# Features 0 and 1 are 'mean radius' and 'mean texture'.

feat_idx  = [0, 1]   # mean radius, mean texture
feat_lbls = [feature_names[i] for i in feat_idx]

X2_train = X_train_scaled[:, feat_idx]
X2_test  = X_test_scaled[:,  feat_idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, kernel in zip(axes, kernels):
    svm2 = SVC(kernel=kernel, random_state=42)
    svm2.fit(X2_train, y_train)

    h  = 0.05
    x_min, x_max = X2_train[:, 0].min() - 0.5, X2_train[:, 0].max() + 0.5
    y_min, y_max = X2_train[:, 1].min() - 0.5, X2_train[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svm2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn')
    scatter = ax.scatter(
        X2_test[:, 0], X2_test[:, 1], c=y_test,
        cmap='RdYlGn', edgecolors='k', lw=0.4, s=40
    )
    acc2 = accuracy_score(y_test, svm2.predict(X2_test))
    ax.set_title(f'SVM — {kernel} kernel\n(2-feature acc: {acc2:.3f})', fontsize=12)
    ax.set_xlabel(feat_lbls[0])

axes[0].set_ylabel(feat_lbls[1])
plt.suptitle('SVM Decision Boundaries (mean radius vs mean texture)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Note: 2-feature accuracy is lower than full-model accuracy —")
print("the remaining 28 features contain valuable signal!")

### Interpretation

The decision boundary visualisation uses only 2 features, so accuracy is lower than the full model. The boundaries illustrate each kernel's characteristic shape:
- **Linear**: a straight line (hyperplane)
- **RBF**: smooth, curved regions that wrap around clusters
- **Poly**: polynomial curves with more inflection points

The RBF kernel almost always performs best on real-world tabular data because it can model any smooth boundary without overfitting (controlled by the `C` and `gamma` parameters).

### SVM — Pros & Cons

| Pros | Cons |
|------|------|
| Excellent generalisation in high-dimensional spaces | Slow on large datasets (O(n²–n³) training) |
| Only support vectors matter — robust to outliers | Requires feature scaling |
| Kernel trick handles non-linear data | Hyperparameters (C, gamma) need tuning |
| Works well with limited training data | Not naturally probabilistic (requires Platt scaling) |
| Strong theoretical foundations | Hard to interpret the decision boundary |

---
## Section 5 — Decision Trees

### How it works

A Decision Tree partitions the feature space by asking a series of yes/no questions:

> *"Is mean radius ≤ 13.1?"*  
> *"Is worst concave points ≤ 0.14?"*  
> …

At each **node**, the algorithm searches for the **best split** — the feature and threshold that most reduces impurity in the resulting subsets.

### Splitting criteria

**Gini impurity** (sklearn default): measures the probability of misclassifying a randomly chosen element.
$$G = 1 - \sum_{k} p_k^2$$

**Information Gain** (entropy-based): measures the reduction in entropy after a split.
$$\text{IG} = H(\text{parent}) - \sum_{\text{child}} \frac{n_\text{child}}{n_\text{parent}} H(\text{child})$$

### Controlling tree depth
Without constraints, a Decision Tree will grow until every leaf contains a single sample — **perfect training accuracy, terrible generalisation**. The `max_depth` parameter is the primary lever to control overfitting.

In [ ]:
# ── Fit Decision Tree (max_depth=5) ──────────────────────────────────────────
# Decision Trees do NOT require scaled features — they split on thresholds,
# not distances. We use the raw (unscaled) train/test splits.

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_result = evaluate_model(
    'Decision Tree (depth=5)', dt_model,
    X_train, X_test, y_train, y_test
)

print(classification_report(y_test, dt_model.predict(X_test), target_names=class_names))

In [ ]:
# ── Visualise the tree (max_depth=3 for readability) ─────────────────────────
dt_vis = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_vis.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(
    dt_vis,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    ax=ax,
    fontsize=9
)
ax.set_title('Decision Tree — First 3 Levels', fontsize=14)
plt.tight_layout()
plt.show()

print("Each node shows: feature threshold | gini | samples | class distribution")

In [ ]:
# ── Feature importance bar chart ──────────────────────────────────────────────
importances = dt_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature'    : feature_names,
    'Importance' : importances
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
bars = plt.barh(
    feat_imp_df['Feature'], feat_imp_df['Importance'],
    color=plt.cm.viridis(np.linspace(0.2, 0.85, len(feat_imp_df)))
)
plt.xlabel('Feature Importance (Gini)', fontsize=12)
plt.title('Decision Tree — Top 15 Feature Importances', fontsize=13)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 5 most important features:")
for _, row in feat_imp_df.head(5).iterrows():
    print(f"  {row['Feature']:<35} {row['Importance']:.4f}")

In [ ]:
# ── Overfitting demo: accuracy vs max_depth (1–20) ────────────────────────────
depths      = range(1, 21)
dt_train_accs = []
dt_test_accs  = []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    dt_train_accs.append(accuracy_score(y_train, clf.predict(X_train)))
    dt_test_accs.append(accuracy_score(y_test,  clf.predict(X_test)))

best_depth = list(depths)[np.argmax(dt_test_accs)]

plt.figure(figsize=(10, 5))
plt.plot(depths, dt_train_accs, 'o-', label='Train accuracy', color='#3498db', lw=2)
plt.plot(depths, dt_test_accs,  's-', label='Test accuracy',  color='#e67e22', lw=2)
plt.axvline(best_depth, ls='--', color='#2ecc71', lw=1.8, label=f'Best depth = {best_depth}')
plt.axvline(5, ls=':', color='gray', lw=1.2, label='Our model (depth=5)')
plt.xlabel('max_depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Decision Tree — Overfitting: Accuracy vs max_depth', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Train accuracy reaches 100% at depth ≥ {next(d for d, a in zip(depths, dt_train_accs) if a == 1.0)}")
print(f"Best test accuracy at depth = {best_depth}  ({max(dt_test_accs):.4f})")

### Interpretation

The overfitting plot makes the bias-variance trade-off concrete:
- At low depth, the tree is too simple to capture patterns (high bias)
- Train accuracy rises steadily with depth
- Test accuracy peaks and then levels off or drops — the tree memorises noise
- Full depth (no limit) gives 100% train accuracy but lower test accuracy

The feature importance chart reveals that **worst features** (worst concave points, worst radius) are more informative than **mean features**. This makes biological sense — the worst-case cell measurements capture the most aggressive cells in the biopsy.

### Decision Tree — Pros & Cons

| Pros | Cons |
|------|------|
| Highly interpretable — can be visualised | Prone to overfitting (needs pruning or depth limit) |
| No feature scaling required | Unstable — small data changes → very different tree |
| Handles mixed feature types natively | Biased towards features with more levels |
| Fast training and prediction | Does not model smooth boundaries well |
| Built-in feature importance | Single tree is outperformed by ensembles (RF, XGBoost) |

---
## Section 6 — Algorithm Comparison

We now compare all four algorithms on the same breast cancer test set. We also re-run each with its best configuration to ensure a fair comparison.

In [ ]:
# ── Build summary DataFrame ───────────────────────────────────────────────────
# Collect the results we logged via evaluate_model()

summary_df = pd.DataFrame([
    {
        'Algorithm'    : r['Algorithm'],
        'Train Acc'    : r['Train Acc'],
        'Test Acc'     : r['Test Acc'],
        'Fit Time (ms)': r['Fit Time']
    }
    for r in results_log
]).sort_values('Test Acc', ascending=False).reset_index(drop=True)

print(summary_df.to_string(index=False))

In [ ]:
# ── Bar chart comparison ──────────────────────────────────────────────────────
algorithms = summary_df['Algorithm']
x = np.arange(len(algorithms))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy comparison
bars1 = axes[0].bar(x - width/2, summary_df['Train Acc'], width,
                    label='Train Acc', color='#3498db', alpha=0.85, edgecolor='white')
bars2 = axes[0].bar(x + width/2, summary_df['Test Acc'],  width,
                    label='Test Acc',  color='#e67e22', alpha=0.85, edgecolor='white')

axes[0].set_xticks(x)
axes[0].set_xticklabels(algorithms, rotation=15, ha='right', fontsize=10)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Train vs Test Accuracy by Algorithm', fontsize=13)
axes[0].set_ylim(0.88, 1.01)
axes[0].legend()

for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

# Fit time comparison (log scale for readability)
axes[1].bar(algorithms, summary_df['Fit Time (ms)'],
            color=['#9b59b6', '#1abc9c', '#e74c3c', '#f39c12'],
            alpha=0.85, edgecolor='white')
axes[1].set_yscale('log')
axes[1].set_xticklabels(algorithms, rotation=15, ha='right', fontsize=10)
axes[1].set_ylabel('Fit Time (ms, log scale)')
axes[1].set_title('Training Time by Algorithm', fontsize=13)

plt.suptitle('Algorithm Comparison — Wisconsin Breast Cancer Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### When to prefer each algorithm

| Scenario | Recommended Algorithm | Reason |
|----------|-----------------------|--------|
| Very small training set (< 100 samples) | Naive Bayes | Low variance, makes strong priors |
| High-dimensional data (> 100 features) | SVM (linear kernel) | Max-margin + L2 regularisation |
| Need for interpretability / explainability | Decision Tree | Can be printed and explained step-by-step |
| Fast prototype / baseline | Naive Bayes or Decision Tree | Near-instant training |
| General-purpose tabular data | SVM (RBF) or KNN | Strong empirical performance |
| Deployment with memory constraints | Decision Tree or Naive Bayes | Minimal parameters to store |
| When probability outputs matter | Naive Bayes or SVM (probability=True) | Calibrated class probabilities |
| Non-linear patterns, moderate-sized data | KNN or SVM (RBF) | Non-parametric / kernel flexibility |

> **Rule of thumb:** Start with Naive Bayes as a fast baseline. Upgrade to SVM (RBF) for best accuracy. Use Decision Tree when stakeholders need to understand the logic.

---
## Section 7 — Hyperparameter Tuning Preview

### What is hyperparameter tuning?

Every algorithm has **hyperparameters** — settings that are not learned from data but set before training. Examples:
- KNN: `n_neighbors` (k)
- SVM: `C` (regularisation), `gamma` (kernel width)
- Decision Tree: `max_depth`, `min_samples_split`

**Grid Search** exhaustively tries every combination of hyperparameter values and uses **cross-validation** to estimate each combination's generalisation performance.

**Cross-validation** splits the training data into *k* folds, trains on *k-1* folds, and validates on the remaining fold — repeated *k* times. This gives a robust estimate of test performance without touching the held-out test set.

> We will cover hyperparameter tuning in depth in **Module 5.5**. This section gives you a preview.

In [ ]:
# ── GridSearchCV on KNN ───────────────────────────────────────────────────────
param_grid = {
    'n_neighbors': list(range(1, 21)),
    'weights'    : ['uniform', 'distance']
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train_scaled, y_train)

print(f"Best parameters : {grid_search.best_params_}")
print(f"Best CV score   : {grid_search.best_score_:.4f}")
print()

# Evaluate best model on held-out test set
best_knn = grid_search.best_estimator_
test_acc_tuned = accuracy_score(y_test, best_knn.predict(X_test_scaled))
print(f"Test accuracy (tuned KNN) : {test_acc_tuned:.4f}")

In [ ]:
# ── Visualise the grid search results ────────────────────────────────────────
cv_results = pd.DataFrame(grid_search.cv_results_)

pivot = cv_results.pivot_table(
    values='mean_test_score',
    index='param_n_neighbors',
    columns='param_weights'
)

pivot.plot(marker='o', figsize=(10, 5), lw=2)
plt.xlabel('n_neighbors (k)', fontsize=12)
plt.ylabel('CV Accuracy (5-fold)', fontsize=12)
plt.title('GridSearchCV — KNN: Cross-Validation Accuracy\nby k and weighting strategy', fontsize=13)
plt.legend(title='weights')
plt.tight_layout()
plt.show()

print("We'll cover GridSearchCV, RandomizedSearchCV, and Bayesian optimisation")
print("in detail in Module 5.5 — Hyperparameter Tuning.")

### Key takeaways from the tuning preview

- `weights='distance'` gives closer neighbours more influence than farther ones — sometimes better, sometimes worse
- Cross-validation gives a more reliable estimate than a single train/test split
- The best CV score is often slightly optimistic compared to the true test set — we use CV to **select** hyperparameters, then report the test set score as our final estimate
- Grid search is exhaustive but slow; we will explore smarter strategies (random search, Bayesian optimisation) in Module 5.5

---
## Section 8 — Practice Exercises

Apply what you've learned. Try each exercise before looking at the solution.

---
### Exercise 1 — KNN with different distance metrics

KNN's default is Euclidean distance (`metric='minkowski'` with `p=2`). Try:
- **Manhattan** distance (`p=1`) — sum of absolute differences, less sensitive to outliers
- **Euclidean** distance (`p=2`) — standard straight-line distance

Use the optimal k found earlier. Which metric gives better test accuracy on this dataset?

In [ ]:
# ── Exercise 1: Your code here ────────────────────────────────────────────────

# Hint: KNeighborsClassifier(n_neighbors=optimal_k, metric='minkowski', p=?)
# Try p=1 (Manhattan) and p=2 (Euclidean)

# YOUR CODE HERE


In [ ]:
# ── Exercise 1: Solution ──────────────────────────────────────────────────────

metrics_to_try = [
    ('Manhattan', {'metric': 'minkowski', 'p': 1}),
    ('Euclidean', {'metric': 'minkowski', 'p': 2}),
    ('Chebyshev', {'metric': 'chebyshev'}),
]

print(f"Using k = {optimal_k}\n")
print(f"{'Metric':<12}  {'Train Acc':>10}  {'Test Acc':>10}")
print("-" * 38)

for name, kwargs in metrics_to_try:
    knn = KNeighborsClassifier(n_neighbors=optimal_k, **kwargs)
    knn.fit(X_train_scaled, y_train)
    tr = accuracy_score(y_train, knn.predict(X_train_scaled))
    te = accuracy_score(y_test,  knn.predict(X_test_scaled))
    print(f"{name:<12}  {tr:>10.4f}  {te:>10.4f}")

---
### Exercise 2 — Decision Tree overfitting (no max_depth)

Train a Decision Tree with **no depth limit** (`max_depth=None`). Compare its train and test accuracy against the depth-5 model. Then inspect the number of leaf nodes. What does this tell you about overfitting?

In [ ]:
# ── Exercise 2: Your code here ────────────────────────────────────────────────

# Hint: DecisionTreeClassifier(max_depth=None, random_state=42)
# Check: model.get_n_leaves() and model.get_depth()

# YOUR CODE HERE


In [ ]:
# ── Exercise 2: Solution ──────────────────────────────────────────────────────

# Unconstrained tree
dt_full = DecisionTreeClassifier(max_depth=None, random_state=42)
dt_full.fit(X_train, y_train)

tr_full = accuracy_score(y_train, dt_full.predict(X_train))
te_full = accuracy_score(y_test,  dt_full.predict(X_test))

# Depth-5 tree for comparison
tr_d5 = accuracy_score(y_train, dt_model.predict(X_train))
te_d5 = accuracy_score(y_test,  dt_model.predict(X_test))

print("Decision Tree comparison:\n")
print(f"{'Model':<22}  {'Depth':>6}  {'Leaves':>7}  {'Train Acc':>10}  {'Test Acc':>10}")
print("-" * 65)
print(f"{'No limit (full)' :<22}  {dt_full.get_depth():>6}  {dt_full.get_n_leaves():>7}  {tr_full:>10.4f}  {te_full:>10.4f}")
print(f"{'max_depth=5'     :<22}  {dt_model.get_depth():>6}  {dt_model.get_n_leaves():>7}  {tr_d5:>10.4f}  {te_d5:>10.4f}")
print()
print("Observation:")
print(f"  Full tree: 100% train acc but {te_full:.4f} test acc — classic overfitting.")
print(f"  Depth-5  : {tr_d5:.4f} train acc, {te_d5:.4f} test acc — better generalisation.")
print(f"  The full tree has {dt_full.get_n_leaves()} leaves for {len(X_train)} training samples — one leaf per ~{len(X_train)/dt_full.get_n_leaves():.1f} samples.")

---
### Exercise 3 — Scaled vs unscaled features for KNN

We said scaling is important for KNN. Verify this experimentally:

1. Train KNN (k=5) on the **raw (unscaled)** features
2. Train KNN (k=5) on the **scaled** features
3. Compare test accuracies

Look at the feature value ranges — why does scaling matter so much here?

In [ ]:
# ── Exercise 3: Your code here ────────────────────────────────────────────────

# Hint: KNeighborsClassifier(n_neighbors=5)
# Train once on X_train (unscaled) and once on X_train_scaled
# Print both test accuracies

# YOUR CODE HERE


In [ ]:
# ── Exercise 3: Solution ──────────────────────────────────────────────────────

# Unscaled KNN
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
te_raw = accuracy_score(y_test, knn_raw.predict(X_test))

# Scaled KNN
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
te_sc = accuracy_score(y_test, knn_scaled.predict(X_test_scaled))

print(f"KNN (k=5) — unscaled features: test accuracy = {te_raw:.4f}")
print(f"KNN (k=5) — scaled features  : test accuracy = {te_sc:.4f}")
print(f"Improvement from scaling      : {(te_sc - te_raw)*100:+.2f} percentage points")
print()

# Show feature value ranges to explain why scaling matters
df_raw = pd.DataFrame(X_train, columns=feature_names)
ranges = (df_raw.max() - df_raw.min()).sort_values(ascending=False)
print("Top 5 features by value range (raw data):")
for fname, rng in ranges.head(5).items():
    print(f"  {fname:<35} range = {rng:.2f}")
print()
print("Without scaling, features with large ranges dominate the distance calculation.")
print("Scaling puts every feature on the same [-3, 3] scale.")

---
## Summary

Congratulations — you've trained and compared four supervised learning algorithms on a real medical dataset!

### What you learned

| Algorithm | Core idea | Best setting | Needs scaling? |
|-----------|-----------|--------------|----------------|
| Naive Bayes | Bayesian probability + independence assumption | Small data, text, fast baseline | No |
| KNN | Vote from k nearest neighbours | Moderate-sized, low-dim data | **Yes** |
| SVM | Maximum-margin hyperplane + kernel trick | High-dim, limited samples | **Yes** |
| Decision Tree | Recursive feature splitting | Interpretability required | No |

### Key concepts to remember

1. **Bias-variance trade-off** — controlling model complexity (k, max_depth) balances underfitting vs overfitting
2. **Feature scaling** — critical for distance-based methods (KNN, SVM), irrelevant for tree-based methods
3. **Data leakage** — always fit scalers on training data only, then apply to test data
4. **No free lunch** — no single algorithm dominates; the best choice depends on data size, dimensionality, and interpretability requirements

### Next steps

- **5.3** — Supervised Learning Part 2: Logistic Regression, Linear Regression, Ridge/Lasso
- **5.4** — Ensemble Methods: Random Forests, Gradient Boosting
- **5.5** — Hyperparameter Tuning: GridSearchCV, RandomizedSearchCV, cross-validation strategies